# Embedding Route

In [15]:
from utils.llm import get_llm, get_embedding_model
from utils.database.qdrant import QdrantIndexManager

llm = get_llm("nvidia-llama3.1")
embed_model = get_embedding_model("BAAI/bge-m3")
vector_store = QdrantIndexManager(embed_model=embed_model, index_name="uu").load_vector_index()

Loaded index 'uu'.


In [16]:
from utils.retriever.uu.embedding_uu import get_embedding_uu

retriever = get_embedding_uu(vector_store=vector_store, llm=llm, embed_model=embed_model, with_self_query=False)

In [18]:
retr = retriever.invoke("jawa barat")

In [21]:
retr[1].metadata

{'title': 'Undang-undang Nomor 10 Tahun 2023 Tentang Provinsi Jawa Barat',
 'jenis_bentuk_peraturan': 'UNDANG-UNDANG',
 'pemrakarsa': 'PEMERINTAH PUSAT',
 'nomor': 10,
 'tahun': 2023,
 'tentang': 'PROVINSI JAWA BARAT',
 'tempat_penetapan': 'Jakarta',
 'ditetapkan_tanggal': '04 Mei 2023',
 'status': 'Berlaku',
 'dokumen_peraturan': 'https://peraturan.go.id/files/uu-no-10-tahun-2023.pdf',
 'id': 'hqel0eiu',
 'filename': 'uu_04_mei_2023_uu_no_10_tahun_2023_hqel0eiu.pdf',
 'page_number': 1,
 '_id': '96aeac5c-241b-4c11-904b-49aee868d0b8',
 '_collection_name': 'uu'}

# Graph Route

In [1]:
from utils.llm import get_llm
from utils.config import get_config

config = get_config()
llm = get_llm("nvidia-llama3.1")

In [2]:
from utils.retriever.uu.graph_uu import get_graph_uu

retriever = get_graph_uu(llm=llm, config=config)

In [3]:
result = retriever.invoke("Undang Undang No 36 Tahun 1953 dicabut peraturan apa saja?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (p:Peraturan {nomor: 36, tahun: 1953, jenis_peraturan: 'UNDANG-UNDANG'})
MATCH (p)-[:DICABUT]->(revoked:Peraturan)
RETURN 
  p AS revokedRegulation, 
  revoked AS revokingRegulation

Full Context:
[]

> Finished chain.


In [4]:
result

{'query': 'Undang Undang No 36 Tahun 1953 dicabut peraturan apa saja?',
 'result': 'Maaf, saya tidak menemukan informasi tentang Undang-Undang No 36 Tahun 1953 dan peraturan yang dicabutnya.'}

# All Together

In [3]:
from utils.llm import get_llm, get_embedding_model
from utils.database.qdrant import QdrantIndexManager
from utils.retriever.uu.embedding_uu import get_embedding_uu

llm = get_llm("nvidia-llama3.1")
embed_model = get_embedding_model("BAAI/bge-m3")
vector_store = QdrantIndexManager(embed_model=embed_model, index_name="uu").load_vector_index()
retriever_uu = get_embedding_uu(vector_store=vector_store, llm=llm, embed_model=embed_model, with_self_query=False)

Loaded index 'uu'.


In [4]:
llm

ChatNVIDIA(base_url='http://localhost:8000/v1', model='meta/llama-3.1-8b-instruct', temperature=0.1, max_tokens=16000)

In [5]:
from utils.retriever.uu.graph_uu import get_graph_uu
from utils.config import get_config

config = get_config()
graph_chain = get_graph_uu(llm=llm, config=config)

In [6]:
from utils.chain.rag import create_combined_answer_chain

chain_quality = create_combined_answer_chain(
  retriever_uu=retriever_uu,
  graph_chain=graph_chain,
  llm=llm
)

In [7]:
response = llm.invoke("what is the capital of jawa barat")

In [8]:
print(response)

content='The capital of Jawa Barat (West Java) is Bandung.' response_metadata={'role': 'assistant', 'content': 'The capital of Jawa Barat (West Java) is Bandung.', 'token_usage': {'prompt_tokens': 21, 'total_tokens': 37, 'completion_tokens': 16}, 'finish_reason': 'stop', 'model_name': 'meta/llama-3.1-8b-instruct'} id='run-c8bd14eb-4daf-43c8-bb41-55f83f2d2dfc-0' role='assistant'


In [9]:
result = chain_quality.invoke({"question": "Undang Undang No 36 Tahun 1953 dicabut peraturan apa saja?", "chat_history": ""})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (p:Peraturan {nomor: '36', tahun: '1953', jenis_peraturan: 'UNDANG-UNDANG'})
MATCH (p)-[:MENCABUT]->(revoked:Peraturan)
RETURN 
  p AS revokingRegulation, 
  revoked AS revokedRegulation

Full Context:
[]

> Finished chain.


In [10]:
print(result['answer'])

Maaf, saya tidak menemukan informasi tentang Undang-Undang No 36 Tahun 1953 yang mencabut peraturan apa saja.


In [11]:
result = chain_quality.invoke({"question": "Apa ibukota dari Provinsi Jawa Barat?", "chat_history": ""})

In [12]:
result

{'rewrited question': 'Apa ibukota dari Provinsi Jawa Barat?',
 'context': [Document(metadata={'title': 'Undang-undang Nomor 10 Tahun 2023 Tentang Provinsi Jawa Barat', 'jenis_bentuk_peraturan': 'UNDANG-UNDANG', 'pemrakarsa': 'PEMERINTAH PUSAT', 'nomor': 10, 'tahun': 2023, 'tentang': 'PROVINSI JAWA BARAT', 'tempat_penetapan': 'Jakarta', 'ditetapkan_tanggal': '04 Mei 2023', 'status': 'Berlaku', 'dokumen_peraturan': 'https://peraturan.go.id/files/uu-no-10-tahun-2023.pdf', 'id': 'hqel0eiu', 'filename': 'uu_04_mei_2023_uu_no_10_tahun_2023_hqel0eiu.pdf', 'page_number': 3, '_id': '007eea41-daf7-4a32-ad0f-7ef0ea342534', '_collection_name': 'uu'}, page_content='- 3 -\nBAB II\nCAKUPAN WILAYAH, IBU KOTA, DAN KARAKTERISTIK\nPROVINSI JAWA BARAT\nPasal 3\nProvinsi Jawa Barat terdiri atas 18 (delapan belas).\nkabupaten dan 9 (sembilan) kota, yaitu:.\n Kabupaten Bogor;\nb.\nKabupaten Sukabumi;\nKabupaten Cianjur;\nd. Kabupaten Bandung;\ne.\nKabupaten Garut;\nf.\nKabupaten Tasikmalaya;\nKabupaten Ciam

In [13]:
print(result['answer'])

### Jawaban:
Ibu kota Provinsi Jawa Barat berkedudukan di Kota Bandung.

### Referensi:
- UU: Ibu kota Provinsi Jawa Barat berkedudukan di Kota Bandung.


In [14]:
result

{'rewrited question': 'Apa ibukota dari Provinsi Jawa Barat?',
 'context': [Document(metadata={'title': 'Undang-undang Nomor 10 Tahun 2023 Tentang Provinsi Jawa Barat', 'jenis_bentuk_peraturan': 'UNDANG-UNDANG', 'pemrakarsa': 'PEMERINTAH PUSAT', 'nomor': 10, 'tahun': 2023, 'tentang': 'PROVINSI JAWA BARAT', 'tempat_penetapan': 'Jakarta', 'ditetapkan_tanggal': '04 Mei 2023', 'status': 'Berlaku', 'dokumen_peraturan': 'https://peraturan.go.id/files/uu-no-10-tahun-2023.pdf', 'id': 'hqel0eiu', 'filename': 'uu_04_mei_2023_uu_no_10_tahun_2023_hqel0eiu.pdf', 'page_number': 3, '_id': '007eea41-daf7-4a32-ad0f-7ef0ea342534', '_collection_name': 'uu'}, page_content='- 3 -\nBAB II\nCAKUPAN WILAYAH, IBU KOTA, DAN KARAKTERISTIK\nPROVINSI JAWA BARAT\nPasal 3\nProvinsi Jawa Barat terdiri atas 18 (delapan belas).\nkabupaten dan 9 (sembilan) kota, yaitu:.\n Kabupaten Bogor;\nb.\nKabupaten Sukabumi;\nKabupaten Cianjur;\nd. Kabupaten Bandung;\ne.\nKabupaten Garut;\nf.\nKabupaten Tasikmalaya;\nKabupaten Ciam